In [1]:
# MODIFIED FROM: phase3-osm-feature-extraction.ipynb
# CHANGES:
#   - Input: prediction_points.csv instead of DHS GPS shapefile
#   - Key column: PointID instead of DHSCLUST
#   - Buffer: Urban_Rural column ('U'=2km, 'R'=5km)
#   - VIIRS merge: uses viirs_ntl_2025_prediction_points.csv
#     with PointID key
#   - Output: static_osm_features_2025_prediction_points.csv
#
# EXPANDED FEATURE SET (updated to match phase3 revised notebook):
#   Buildings: 4 metrics per type (count, total area, mean area,
#     proportion) + 5 aggregate features (total count, total area,
#     mean area, proportion, density per km²)
#   POIs: 15 typed categories (corrected fclass values, 4 new
#     categories: restaurant, police, post_office, community, sports)
#   Landuse: 5 area features (residential, commercial, industrial,
#     agricultural, forest) if landuse shapefile present
#   Total: 52 feature columns + PointID + VIIRS_Median = 53 columns

import geopandas as gpd
import pandas as pd
import numpy as np
import os
import warnings
from shapely.geometry import Point
from datetime import datetime
warnings.filterwarnings('ignore')

print('Imports ready.')

Imports ready.


In [2]:
# ==========================================
# 1. SETUP PATHS
# ==========================================
# CHANGED:
#   - prediction_points_csv replaces dhs_shp_path
#   - viirs_csv_path points to 2025 prediction points VIIRS
#   - output_csv is a new file for inference
# UNCHANGED:
#   - base_osm_dir (same OSM shapefile as training)

# OSM shapefile folder (same as training)
base_osm_dir = '/Users/ruben/Desktop/Thesis/TrainingData/PH_OSM.shp'

# CHANGED: prediction points CSV instead of DHS GPS shapefile
prediction_points_csv = '/Users/ruben/Desktop/Thesis/2025Data/prediction_points.csv'

# CHANGED: 2025 VIIRS for prediction points (output of phase5-3)
viirs_csv_path = '/Users/ruben/Desktop/Thesis/2025Data/viirs_ntl_2025_prediction_points.csv'

# CHANGED: output file for inference
output_csv = '/Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_expanded.csv'

# Verify all input files exist
for label, path in [
    ('OSM dir',           base_osm_dir),
    ('Prediction points', prediction_points_csv),
    ('VIIRS 2025',        viirs_csv_path)
]:
    status = '\u2713' if os.path.exists(path) else '\u2717 MISSING'
    print(f'  {status}  {label}: {path}')

  ✓  OSM dir: /Users/ruben/Desktop/Thesis/TrainingData/PH_OSM.shp
  ✓  Prediction points: /Users/ruben/Desktop/Thesis/2025Data/prediction_points.csv
  ✓  VIIRS 2025: /Users/ruben/Desktop/Thesis/2025Data/viirs_ntl_2025_prediction_points.csv


In [3]:
# ==========================================
# 2. LOAD DATA
# ==========================================
# CHANGED: loads prediction_points.csv and converts to
# GeoDataFrame from Latitude/Longitude columns.
# The resulting gdf_points has the same structure as the
# original gdf_dhs but keyed on PointID and Urban_Rural.
# UNCHANGED: OSM shapefile loading logic.

print('Loading prediction points CSV...')
df_pts = pd.read_csv(prediction_points_csv)
print(f'Prediction points: {len(df_pts)}')
print(f'Columns: {list(df_pts.columns)}')
print(f'Province distribution:')
print(df_pts['Province'].value_counts().to_string())

# Convert to GeoDataFrame using Latitude/Longitude
geometry    = [Point(xy) for xy in
               zip(df_pts['Longitude'], df_pts['Latitude'])]
gdf_points  = gpd.GeoDataFrame(
    df_pts, geometry=geometry, crs='EPSG:4326')
print(f'\nGeoDataFrame created: {len(gdf_points)} points')

# Load OSM files (unchanged from training)
print('\nLoading OSM shapefiles...')
try:
    gdf_roads = gpd.read_file(
        os.path.join(base_osm_dir, 'gis_osm_roads_free_1.shp'))
    gdf_bldgs = gpd.read_file(
        os.path.join(base_osm_dir,
                     'gis_osm_buildings_a_free_1.shp'))
    gdf_pois  = gpd.read_file(
        os.path.join(base_osm_dir, 'gis_osm_pois_free_1.shp'))
    print('\u2713 Loaded Roads, Buildings, and POIs.')
    print(f'  Roads     : {len(gdf_roads):,} features')
    print(f'  Buildings : {len(gdf_bldgs):,} features')
    print(f'  POIs      : {len(gdf_pois):,} features')
except Exception as e:
    print(f'Error loading OSM files: {e}')
    print('Ensure the OSM shapefile folder exists.')

Loading prediction points CSV...
Prediction points: 1240
Columns: ['PointID', 'Latitude', 'Longitude', 'Municipality', 'Province', 'Region', 'Urban_Rural', 'source', 'DHSCLUST', 'PSGC', 'PSA_Poverty_Rate']
Province distribution:
Province
Zamboanga del Norte    239
Davao Oriental         194
NCR                    149
Ilocos Norte           119
Kalinga                116
Benguet                107
Pampanga                84
Maguindanao del Sur     83
Aklan                   61
Basilan                 44
Tawi-Tawi               44

GeoDataFrame created: 1240 points

Loading OSM shapefiles...
✓ Loaded Roads, Buildings, and POIs.
  Roads     : 1,584,797 features
  Buildings : 11,730,420 features
  POIs      : 168,727 features


In [4]:
# ==========================================
# 3. REPROJECT AND BUFFER
# ==========================================
# CHANGED: buffer logic now uses Urban_Rural column from
# prediction_points.csv instead of URBAN_RURA from shapefile.
#   Urban_Rural == 'U' -> 2,000m (NCR)
#   Urban_Rural == 'R' -> 5,000m (all other provinces)
# This exactly mirrors the training buffer protocol.
# UNCHANGED: target CRS, buffer distances.

target_crs = 'EPSG:32651'

print('Reprojecting to EPSG:32651...')
gdf_points = gdf_points.to_crs(target_crs)
gdf_roads  = gdf_roads.to_crs(target_crs)
gdf_bldgs  = gdf_bldgs.to_crs(target_crs)
gdf_pois   = gdf_pois.to_crs(target_crs)

# CHANGED: use Urban_Rural column (was URBAN_RURA)
print('Creating adaptive buffers...')
def get_buffer(row):
    if str(row.get('Urban_Rural', 'R')).upper() == 'U':
        return row.geometry.buffer(2000)  # 2km for Urban (NCR)
    else:
        return row.geometry.buffer(5000)  # 5km for Rural

gdf_points['buffer_geom'] = gdf_points.apply(
    get_buffer, axis=1)

# Verify buffer sizes
urban_count = (gdf_points['Urban_Rural'].str.upper() == 'U').sum()
rural_count = (gdf_points['Urban_Rural'].str.upper() == 'R').sum()
print(f'Urban points (2km buffer): {urban_count}')
print(f'Rural points (5km buffer): {rural_count}')
print('\u2713 Buffers created')

Reprojecting to EPSG:32651...
Creating adaptive buffers...
Urban points (2km buffer): 149
Rural points (5km buffer): 1091
✓ Buffers created


In [5]:
# ==========================================
# 4. LOAD OPTIONAL LANDUSE LAYER
# ==========================================
# Unchanged from revised phase3 notebook.
# PointID/DHSCLUST distinction does not affect this cell.

landuse_path = os.path.join(
    base_osm_dir, 'gis_osm_landuse_a_free_1.shp')

if os.path.exists(landuse_path):
    gdf_landuse = gpd.read_file(landuse_path).to_crs(target_crs)
    lu_sindex   = gdf_landuse.sindex
    HAS_LANDUSE = True
    print(f'Landuse layer loaded: {len(gdf_landuse):,} polygons')
    print('Landuse classes present:')
    print(gdf_landuse['fclass'].value_counts().head(15).to_string())
else:
    HAS_LANDUSE = False
    print('WARNING: gis_osm_landuse_a_free_1.shp not found.')
    print('  All LU_* features will be 0.')
    print('  Download from: download.geofabrik.de/asia/philippines.html')

Landuse layer loaded: 257,531 polygons
Landuse classes present:
fclass
residential          100320
farmland              70981
forest                21957
orchard               12322
industrial             9017
scrub                  7187
retail                 7170
grass                  6702
commercial             6232
park                   5077
cemetery               4116
farmyard               2290
meadow                 1959
recreation_ground       884
quarry                  572


In [6]:
# ==========================================
# 5. FEATURE ENGINEERING LOOP
# ==========================================
# CHANGED from original phase6 notebook:
#   - Expanded building metrics: 4 per type (count, total area,
#     mean area, proportion) + 5 aggregate features
#   - Extended POI categories: 15 typed (corrected fclass values)
#   - Landuse features: 5 area columns (guarded by HAS_LANDUSE)
#   - Derived features: Mean_Bldg_Area, Bldg_Density_per_km2
# UNCHANGED:
#   - Key column: PointID (not DHSCLUST)
#   - Iterates over gdf_points (not gdf_dhs)
#   - Urban_Rural buffer logic (not URBAN_RURA)
#   - Road extraction logic
#   - Checkpoint interval and naming

print('Extracting OSM features for prediction points...')
results = []

road_sindex = gdf_roads.sindex
bldg_sindex = gdf_bldgs.sindex
poi_sindex  = gdf_pois.sindex

# ── Road type mapping ──────────────────────────────────────────────
ROAD_TYPES = {
    'Main_Roads'     : ['motorway', 'trunk',
                        'primary', 'primary_link'],
    'Secondary_Roads': ['secondary', 'tertiary'],
    'Local_Roads'    : ['residential', 'living_street',
                        'unclassified', 'service'],
    'Tracks'         : ['track', 'path'],
}

# ── Building type mapping ──────────────────────────────────────────
BLDG_TYPES = {
    'residential': ['residential', 'house',
                    'apartments', 'detached'],
    'commercial' : ['commercial', 'retail',
                    'office', 'supermarket'],
    'industrial' : ['industrial', 'warehouse', 'factory'],
    'school'     : ['school', 'university',
                    'college', 'kindergarten'],
    'hospital'   : ['hospital', 'clinic', 'health_centre'],
}

# ── POI type mapping (verified fclass values, PH Geofabrik 2026) ───
# Dropped from original: fuel, ferry_terminal, bus_stop/bus_station,
#   place_of_worship (0 records in PH shapefile).
# Corrected: guesthouse (was guest_house), market_place (was
#   marketplace), town_hall (was townhall).
POI_TYPES = {
    # Financial
    'bank'       : ['bank'],
    'atm'        : ['atm'],
    # Accommodation
    'hotel'      : ['hotel', 'motel', 'guesthouse', 'hostel'],
    # Food & retail
    'fast_food'  : ['fast_food'],
    'convenience': ['convenience'],
    'restaurant' : ['restaurant', 'cafe', 'food_court'],
    'market'     : ['market_place', 'supermarket'],
    # Education
    'school'     : ['school', 'college', 'university',
                    'kindergarten'],
    # Health
    'hospital'   : ['hospital', 'clinic', 'doctors'],
    'pharmacy'   : ['pharmacy'],
    # Civic & government
    'government' : ['town_hall', 'public_building'],
    'police'     : ['police', 'fire_station'],
    'post_office': ['post_office'],
    'community'  : ['community_centre'],
    # Leisure
    'sports'     : ['pitch', 'sports_centre', 'swimming_pool'],
}

# ── Landuse class mapping ──────────────────────────────────────────
LU_TYPES = {
    'LU_Residential_m2' : ['residential'],
    'LU_Commercial_m2'  : ['commercial', 'retail'],
    'LU_Industrial_m2'  : ['industrial'],
    'LU_Agricultural_m2': ['farmland', 'orchard',
                            'vineyard', 'allotments'],
    'LU_Forest_m2'      : ['forest', 'wood'],
}

# CHANGED: iterate over gdf_points, key on PointID
for loop_idx, (idx, row) in enumerate(gdf_points.iterrows()):

    point_id = row['PointID']    # CHANGED: was DHSCLUST
    buffer   = row['buffer_geom']
    n_done   = loop_idx + 1

    # Buffer area for proportion / density calculations.
    # CHANGED: reads Urban_Rural (not URBAN_RURA)
    is_urban       = str(row.get('Urban_Rural', 'R')).upper() == 'U'
    buffer_area_m2 = np.pi * (2000**2 if is_urban else 5000**2)
    buffer_area_km2 = buffer_area_m2 / 1e6

    rec = {'PointID': point_id}    # CHANGED: was DHSCLUST

    # ── ROADS ──────────────────────────────────────────────────────
    possible_roads = gdf_roads.iloc[
        list(road_sindex.intersection(buffer.bounds))]
    precise_roads  = possible_roads[
        possible_roads.intersects(buffer)]

    if len(precise_roads) > 0:
        clipped_roads = precise_roads.geometry.intersection(buffer)
        rec['Total_Road_Length'] = (
            clipped_roads.length.sum() / 1000.0)
        for r_cat, r_classes in ROAD_TYPES.items():
            mask = precise_roads['fclass'].isin(r_classes)
            rec[f'{r_cat}_Length'] = (
                clipped_roads[mask].length.sum() / 1000.0)
    else:
        rec['Total_Road_Length'] = 0.0
        for r_cat in ROAD_TYPES:
            rec[f'{r_cat}_Length'] = 0.0

    # ── BUILDINGS ──────────────────────────────────────────────────
    # Clip all buildings once, then filter by type.
    possible_bldgs = gdf_bldgs.iloc[
        list(bldg_sindex.intersection(buffer.bounds))]
    precise_bldgs  = possible_bldgs[
        possible_bldgs.intersects(buffer)].copy()

    if len(precise_bldgs) > 0:
        precise_bldgs['clip_geom'] = (
            precise_bldgs.geometry.intersection(buffer))
        precise_bldgs['clip_area'] = precise_bldgs['clip_geom'].area

        type_col = ('type' if 'type' in precise_bldgs.columns
                    else 'fclass')

        total_count = len(precise_bldgs)
        total_area  = float(precise_bldgs['clip_area'].sum())
        mean_area   = float(precise_bldgs['clip_area'].mean())
        proportion  = round(total_area / buffer_area_m2, 6)

        rec['Total_Bldg_Count']      = total_count
        rec['Total_Bldg_Area']       = round(total_area, 2)
        rec['Mean_Bldg_Area']        = round(mean_area, 2)
        rec['Total_Bldg_Proportion'] = proportion
        rec['Bldg_Density_per_km2']  = round(
            total_count / buffer_area_km2, 4)

        for t_name, t_tags in BLDG_TYPES.items():
            subset  = precise_bldgs[
                precise_bldgs[type_col].isin(t_tags)]
            t_count = len(subset)
            t_area  = float(subset['clip_area'].sum())
            t_mean  = float(
                subset['clip_area'].mean() if t_count > 0 else 0.0)
            t_prop  = round(t_area / buffer_area_m2, 6)

            rec[f'Bldg_{t_name}_Count']      = t_count
            rec[f'Bldg_{t_name}_TotalArea']  = round(t_area, 2)
            rec[f'Bldg_{t_name}_MeanArea']   = round(t_mean, 2)
            rec[f'Bldg_{t_name}_Proportion'] = t_prop
    else:
        rec['Total_Bldg_Count']      = 0
        rec['Total_Bldg_Area']       = 0.0
        rec['Mean_Bldg_Area']        = 0.0
        rec['Total_Bldg_Proportion'] = 0.0
        rec['Bldg_Density_per_km2']  = 0.0
        for t_name in BLDG_TYPES:
            rec[f'Bldg_{t_name}_Count']      = 0
            rec[f'Bldg_{t_name}_TotalArea']  = 0.0
            rec[f'Bldg_{t_name}_MeanArea']   = 0.0
            rec[f'Bldg_{t_name}_Proportion'] = 0.0

    # ── POIs (extended) ────────────────────────────────────────────
    possible_pois = gdf_pois.iloc[
        list(poi_sindex.intersection(buffer.bounds))]
    precise_pois  = possible_pois[
        possible_pois.intersects(buffer)]

    rec['Total_POI_Count'] = len(precise_pois)

    if len(precise_pois) > 0:
        poi_counts = precise_pois['fclass'].value_counts()
        for p_name, p_tags in POI_TYPES.items():
            rec[f'POI_{p_name}_Count'] = int(
                sum(poi_counts.get(tag, 0) for tag in p_tags))
    else:
        for p_name in POI_TYPES:
            rec[f'POI_{p_name}_Count'] = 0

    # ── LANDUSE (optional) ─────────────────────────────────────────
    if HAS_LANDUSE:
        possible_lu = gdf_landuse.iloc[
            list(lu_sindex.intersection(buffer.bounds))]
        precise_lu  = possible_lu[
            possible_lu.intersects(buffer)].copy()

        if len(precise_lu) > 0:
            precise_lu['clip_area'] = (
                precise_lu.geometry.intersection(buffer).area)
            for lu_col, lu_tags in LU_TYPES.items():
                area = precise_lu[
                    precise_lu['fclass'].isin(lu_tags)
                ]['clip_area'].sum()
                rec[lu_col] = round(float(area), 2)
        else:
            for lu_col in LU_TYPES:
                rec[lu_col] = 0.0
    else:
        for lu_col in LU_TYPES:
            rec[lu_col] = 0.0

    results.append(rec)

    # ── Progress + checkpoint ──────────────────────────────────────
    if n_done % 50 == 0 or n_done == len(gdf_points):
        print(f'  [{datetime.now().strftime("%H:%M:%S")}] '
              f'{n_done}/{len(gdf_points)} points done')
    if n_done % 200 == 0:
        cp = output_csv.replace('.csv',
                                f'_checkpoint_{n_done}.csv')
        pd.DataFrame(results).to_csv(cp, index=False)
        print(f'  Checkpoint saved: {cp}')

print(f'\n\u2713 Extraction complete. {len(results)} records.')

Extracting OSM features for prediction points...
  [15:55:15] 50/1240 points done
  [15:55:16] 100/1240 points done
  [15:55:29] 150/1240 points done
  [15:55:40] 200/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_expanded_checkpoint_200.csv
  [15:55:42] 250/1240 points done
  [15:55:43] 300/1240 points done
  [15:55:43] 350/1240 points done
  [15:55:44] 400/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_expanded_checkpoint_400.csv
  [15:55:44] 450/1240 points done
  [15:55:45] 500/1240 points done
  [15:55:45] 550/1240 points done
  [15:55:45] 600/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_expanded_checkpoint_600.csv
  [15:55:50] 650/1240 points done
  [15:55:53] 700/1240 points done
  [15:55:55] 750/1240 points done
  [15:55:55] 800/1240 points done
  Checkpoint saved: /Use

In [8]:
# ==========================================
# 6. MERGE VIIRS AND SAVE
# ==========================================
# CHANGED:
#   - Merges on PointID instead of DHSCLUST
#   - EXPECTED_OSM_COLS updated to 52 features
#   - Summary prints feature category breakdown
# UNCHANGED: VIIRS_Median column, merge strategy, fillna.

df_osm = pd.DataFrame(results)
df_osm['PointID'] = df_osm['PointID'].astype(int)

print(f'OSM features extracted : {len(df_osm)} points')
print(f'Total columns          : {len(df_osm.columns)}')

# Load 2025 VIIRS
print(f'\nLoading VIIRS 2025 from: {viirs_csv_path}')
df_viirs = pd.read_csv(viirs_csv_path)
df_viirs['PointID'] = df_viirs['PointID'].astype(int)

if 'VIIRS_Median' not in df_viirs.columns:
    for col in df_viirs.columns:
        if col.lower() in ('median', 'ntl_value',
                           'avg_rad', 'viirs_median'):
            df_viirs = df_viirs.rename(
                columns={col: 'VIIRS_Median'})
            break

df_viirs_clean = df_viirs[['PointID', 'VIIRS_Median']].copy()

osm_ids   = set(df_osm['PointID'])
viirs_ids = set(df_viirs_clean['PointID'])
missing   = osm_ids - viirs_ids
if missing:
    print(f'WARNING: {len(missing)} points missing VIIRS.')
else:
    print(f'\u2713 Full VIIRS coverage for all {len(osm_ids)} points.')

df_final = pd.merge(
    df_osm, df_viirs_clean, on='PointID', how='left')
df_final['VIIRS_Median'] = df_final['VIIRS_Median'].fillna(0)
df_final = df_final.sort_values(
    'PointID').reset_index(drop=True)

# Verify column schema matches training
EXPECTED_OSM_COLS = ['Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length', 'Total_Bldg_Count', 'Total_Bldg_Area', 'Mean_Bldg_Area', 'Total_Bldg_Proportion', 'Bldg_Density_per_km2', 'Bldg_residential_Count', 'Bldg_residential_TotalArea', 'Bldg_residential_MeanArea', 'Bldg_residential_Proportion', 'Bldg_commercial_Count', 'Bldg_commercial_TotalArea', 'Bldg_commercial_MeanArea', 'Bldg_commercial_Proportion', 'Bldg_industrial_Count', 'Bldg_industrial_TotalArea', 'Bldg_industrial_MeanArea', 'Bldg_industrial_Proportion', 'Bldg_school_Count', 'Bldg_school_TotalArea', 'Bldg_school_MeanArea', 'Bldg_school_Proportion', 'Bldg_hospital_Count', 'Bldg_hospital_TotalArea', 'Bldg_hospital_MeanArea', 'Bldg_hospital_Proportion', 'Total_POI_Count', 'POI_bank_Count', 'POI_atm_Count', 'POI_hotel_Count', 'POI_fast_food_Count', 'POI_convenience_Count', 'POI_restaurant_Count', 'POI_market_Count', 'POI_school_Count', 'POI_hospital_Count', 'POI_pharmacy_Count', 'POI_government_Count', 'POI_police_Count', 'POI_post_office_Count', 'POI_community_Count', 'POI_sports_Count', 'LU_Residential_m2', 'LU_Commercial_m2', 'LU_Industrial_m2', 'LU_Agricultural_m2', 'LU_Forest_m2', 'VIIRS_Median']

missing_cols = [c for c in EXPECTED_OSM_COLS
                if c not in df_final.columns]
if missing_cols:
    print(f'WARNING: Missing columns: {missing_cols}')
else:
    print(f'\u2713 All 52 static feature columns present.')
    print(f'  Schema matches revised training'
          f' static_osm_features_full_expanded.csv')

# Save
df_final.to_csv(output_csv, index=False)

# ── Summary ───────────────────────────────────────────────────────
all_cols  = list(df_final.columns)
road_cols = [c for c in all_cols if 'Road' in c or 'Track' in c]
bldg_cols = [c for c in all_cols
             if 'Bldg' in c or c in (
                 'Total_Bldg_Count','Total_Bldg_Area',
                 'Mean_Bldg_Area','Total_Bldg_Proportion')]
poi_cols  = [c for c in all_cols
             if c.startswith('POI_')
             or c == 'Total_POI_Count']
lu_cols   = [c for c in all_cols if c.startswith('LU_')]

print(f'\n{"="*50}')
print('EXTRACTION SUMMARY')
print(f'{"="*50}')
print(f'Points extracted : {len(df_final)}')
print(f'Total columns    : {len(df_final.columns)}')
print(f'  Road features  : {len(road_cols)}')
print(f'  Bldg features  : {len(bldg_cols)}')
print(f'  POI features   : {len(poi_cols)}')
print(f'  Landuse feat.  : {len(lu_cols)}')
print(f'  VIIRS          : 1')
print(f'Missing values   : {df_final.isnull().sum().sum()}')
print(f'VIIRS zeros      : {(df_final["VIIRS_Median"]==0).sum()}')
print(f'\nSaved to: {output_csv}')

OSM features extracted : 1240 points
Total columns          : 52

Loading VIIRS 2025 from: /Users/ruben/Desktop/Thesis/2025Data/viirs_ntl_2025_prediction_points.csv
✓ Full VIIRS coverage for all 1240 points.
✓ All 52 static feature columns present.
  Schema matches revised training static_osm_features_full_expanded.csv

EXTRACTION SUMMARY
Points extracted : 1240
Total columns    : 53
  Road features  : 5
  Bldg features  : 25
  POI features   : 16
  Landuse feat.  : 5
  VIIRS          : 1
Missing values   : 0
VIIRS zeros      : 0

Saved to: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_expanded.csv
